# 005 Rule-Based Triage Safety Baseline

Inspect the first deterministic routing and safety baseline on dev fixtures only. This notebook is a review surface; it should not train, index, load models, or generate troubleshooting answers.

## Stage Contract

- Use only question-only eval fixtures from `004`.
- Keep holdout files untouched during dev tuning.
- Compare `text_only` and `metadata_assisted` profiles honestly.
- Treat Stack Exchange tags as source metadata, not normal user-ticket text.
- Record safety false negatives for manual review before claiming safety performance.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

plt.rcParams.update({'figure.figsize': (9, 5), 'axes.grid': True})

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

OUT = ROOT / 'data' / 'eval' / 'rule_based_triage_safety_baseline'
SCRIPT = ROOT / 'scripts' / 'evaluate_rule_based_triage_safety.py'
REVIEW_SCRIPT = ROOT / 'scripts' / 'review_rule_based_safety_false_negatives.py'

SUMMARY = OUT / 'baseline_summary_dev_only.json'
REVIEW_SUMMARY = OUT / 'safety_false_negative_review_summary_dev_only.md'

## Regenerate Dev Baseline And Review

Leave `RUN_EVAL` and `RUN_REVIEW` false unless you are intentionally refreshing the fast dev-only baseline or its false-negative review. Do not pass `--include-holdout` during dev tuning.

In [ ]:
RUN_EVAL = False

if RUN_EVAL:
    result = subprocess.run(
        [sys.executable, str(SCRIPT), '--summary'],
        cwd=ROOT,
        check=True,
        text=True,
        capture_output=True,
    )
    print(result.stdout)

RUN_REVIEW = False

if RUN_REVIEW:
    result = subprocess.run(
        [sys.executable, str(REVIEW_SCRIPT), '--summary'],
        cwd=ROOT,
        check=True,
        text=True,
        capture_output=True,
    )
    print(result.stdout)

assert SUMMARY.exists(), f'Missing baseline summary: {SUMMARY}'
assert REVIEW_SUMMARY.exists(), f'Missing safety false-negative review: {REVIEW_SUMMARY}'

## Summary

In [ ]:
display(Markdown((OUT / 'baseline_summary_dev_only.md').read_text(encoding='utf-8')))
summary = json.loads(SUMMARY.read_text(encoding='utf-8'))
summary['input_counts']

## Routing Metrics

In [ ]:
routing_metrics = pd.read_csv(OUT / 'routing_metrics_dev_only.csv')
routing_f1 = routing_metrics.pivot(index='domain', columns='profile', values='f1')
display(routing_f1)

ax = routing_f1.sort_values('metadata_assisted').plot.barh()
ax.set_title('Routing F1 by Domain')
ax.set_xlabel('F1')
ax.set_ylabel('')
ax.set_xlim(0, 1)
plt.tight_layout()
plt.show()

routing_confusion = pd.read_csv(OUT / 'routing_primary_confusion_dev_only.csv')
profile = 'metadata_assisted'
routing_matrix = routing_confusion[routing_confusion['profile'].eq(profile)].pivot_table(
    index='expected_primary_domain',
    columns='predicted_primary_domain',
    values='records',
    aggfunc='sum',
    fill_value=0,
)
display(routing_matrix)

fig, ax = plt.subplots(figsize=(8, 6))
image = ax.imshow(routing_matrix.to_numpy(), aspect='auto')
ax.set_title(f'Primary-Domain Confusion: {profile}')
ax.set_xlabel('Predicted')
ax.set_ylabel('Expected')
ax.set_xticks(range(len(routing_matrix.columns)), routing_matrix.columns, rotation=45, ha='right')
ax.set_yticks(range(len(routing_matrix.index)), routing_matrix.index)
fig.colorbar(image, ax=ax, label='records')
plt.tight_layout()
plt.show()

display(routing_metrics.sort_values(['profile', 'f1']))

## Safety Metrics

In [ ]:
safety_metrics = pd.read_csv(OUT / 'safety_metrics_dev_only.csv')
display(safety_metrics)

safety_f1 = safety_metrics.pivot(index='expected_behavior', columns='profile', values='f1')
ax = safety_f1.plot.bar(rot=20)
ax.set_title('Safety Behavior F1')
ax.set_xlabel('')
ax.set_ylabel('F1')
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

safety_confusion = pd.read_csv(OUT / 'safety_confusion_dev_only.csv')
profile = 'metadata_assisted'
safety_matrix = safety_confusion[safety_confusion['profile'].eq(profile)].pivot_table(
    index='expected_behavior',
    columns='predicted_behavior',
    values='records',
    aggfunc='sum',
    fill_value=0,
)
display(safety_matrix)

fig, ax = plt.subplots(figsize=(8, 4))
image = ax.imshow(safety_matrix.to_numpy(), aspect='auto')
ax.set_title(f'Safety Confusion: {profile}')
ax.set_xlabel('Predicted')
ax.set_ylabel('Expected')
ax.set_xticks(range(len(safety_matrix.columns)), safety_matrix.columns, rotation=35, ha='right')
ax.set_yticks(range(len(safety_matrix.index)), safety_matrix.index)
fig.colorbar(image, ax=ax, label='records')
plt.tight_layout()
plt.show()

## False Negative Samples

In [ ]:
false_negatives = pd.read_csv(OUT / 'safety_false_negatives_dev_only.csv')
tag_counts = pd.read_csv(OUT / 'safety_false_negative_tag_counts_dev_only.csv')

display(false_negatives.groupby(['profile', 'expected_behavior']).size().rename('records'))
display(false_negatives.head(30))

profile = 'metadata_assisted'
top_tags = (
    tag_counts[tag_counts['profile'].eq(profile)]
    .groupby('question_tag', as_index=False)['records']
    .sum()
    .sort_values('records', ascending=False)
    .head(20)
    .sort_values('records')
)
if top_tags.empty:
    display(Markdown(f'No safety false-negative tags for `{profile}`.'))
else:
    ax = top_tags.plot.barh(x='question_tag', y='records', legend=False)
    ax.set_title(f'Top Safety False-Negative Tags: {profile}')
    ax.set_xlabel('records')
    ax.set_ylabel('')
    plt.tight_layout()
    plt.show()

## False Negative Review

In [ ]:
display(Markdown(REVIEW_SUMMARY.read_text(encoding='utf-8')))

review_counts = pd.read_csv(OUT / 'safety_false_negative_review_bucket_counts_dev_only.csv')
display(review_counts)

if review_counts.empty:
    display(Markdown('No metadata-assisted safety false negatives remain in the dev-only baseline.'))
else:
    ax = review_counts.sort_values('records').plot.barh(x='review_bucket', y='records', legend=False)
    ax.set_title('Safety False-Negative Review Buckets')
    ax.set_xlabel('records')
    ax.set_ylabel('')
    plt.tight_layout()
    plt.show()

review_expected = pd.read_csv(OUT / 'safety_false_negative_review_expected_bucket_counts_dev_only.csv')
if not review_expected.empty:
    display(review_expected.pivot_table(
        index='review_bucket',
        columns='expected_behavior',
        values='records',
        aggfunc='sum',
        fill_value=0,
    ))

## Prediction Samples

In [ ]:
predictions = pd.read_json(OUT / 'routing_predictions_dev_only_text_only.jsonl', lines=True)
predictions[['case_id', 'title', 'expected_primary_domain', 'predicted_primary_domain', 'predicted_domains']].head(20)

## Exit Criteria

- Baseline report exists for dev fixtures only.
- No holdout files were read during normal notebook inspection.
- No models, embeddings, FAISS indexes, training data, or answer-generation records were produced.
- Per-domain routing metrics, safety false-negative counts, and review buckets are visible.
- Next step is targeted dev-only rule/gate refinement based on review buckets before any holdout run.